In [1]:
!pip install vllm

In [46]:
from google.colab import drive, userdata
import json
import os
import pandas as pd
from tqdm.auto import tqdm
from pydantic import BaseModel, Field, model_validator
from vllm import SamplingParams
from vllm.sampling_params import StructuredOutputsParams
from datasets import Dataset, DatasetDict, load_dataset
from transformers import AutoTokenizer, BartTokenizer

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
login(token=hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [5]:
splits = ['Train', 'Val', 'Test']
dfs = {}

for split in splits:
  path = f'/content/drive/MyDrive/266_Final/data/{split}/data.parquet'
  dfs[split] = pd.read_parquet(path)
  print(f'{split} shape: {dfs[split].shape}')

train_df, val_df, test_df = dfs['Train'], dfs['Val'], dfs['Test']

Train shape: (53492, 6)
Val shape: (6686, 6)
Test shape: (6687, 6)


In [22]:
class ExtractedTable(BaseModel):
    table_name: str = Field(description="Descriptive title or section context for the table")
    headers: list[str]
    rows: list[list[str]]

    @model_validator(mode="after")
    def normalize_row_widths(self) -> "ExtractedTable":
        width = len(self.headers)
        if width == 0:
            self.rows = []
            return self
        normalized = []
        for row in self.rows:
            if len(row) < width:
                normalized.append(row + [""] * (width - len(row)))
            elif len(row) > width:
                normalized.append(row[: width - 1] + [" ".join(row[width - 1 :])])
            else:
                normalized.append(row)
        self.rows = normalized
        return self

class DocumentTables(BaseModel):
    tables: list[ExtractedTable]

In [9]:
llm = LLM(
    model="meta-llama/Llama-3.3-70B-Instruct",
    quantization="fp8",       # Compresses 141 GB -> ~71 GB so it fits in 95 GB VRAM
    dtype="bfloat16",
    max_model_len=8192,       # Prevents vLLM from reserving 131,072 tokens of KV cache
    gpu_memory_utilization=0.90,
)

(MainProcess pid=3073) INFO 11-14 03:53:03 [startup.py:96] non-default args: {'dtype': 'bfloat16', 'max_model_len': 8192, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'quantization': 'fp8', 'model': 'meta-llama/Llama-3.3-70B-Instruct'}
(MainProcess pid=3073) INFO 11-14 03:53:03 [model.py:744] Resolved architecture: LlamaForCausalLM
(MainProcess pid=3073) INFO 11-14 03:53:03 [model.py:2170] Using max model len 8192
(MainProcess pid=3073) WARNING 11-14 03:53:04 [weight_utils.py:357] --quantization fp8 is deprecated for online quantization; use --quantization fp8_per_tensor instead.
(MainProcess pid=3073) INFO 11-14 03:53:04 [kernel.py:480] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
(EngineCore pid=32543) INFO 11-14 03:53:05 [core.py:139] Initializing a V1 LLM engine (v0.33.0) with config: model='meta-llama/Llama-3.3-70B-Instruct', speculative_config=Non

Loading safetensors checkpoint shards:   0% Completed | 0/30 [00:00<?, ?it/s]


(EngineCore pid=32543) INFO 11-14 04:06:59 [default_loader.py:506] Loading weights took 411.77 seconds
(EngineCore pid=32543) INFO 11-14 04:06:59 [base_loader.py:164] Online quantization as part of model loading took 0.36 seconds
(EngineCore pid=32543) INFO 11-14 04:07:01 [model_runner.py:428] Model loading took 67.7 GiB memory and 835.051392 seconds
(EngineCore pid=32543) INFO 11-14 04:07:01 [topk_topp_sampler.py:149] Using FlashInfer for top-p & top-k sampling.
(EngineCore pid=32543) INFO 11-14 04:07:01 [torch_utils.py:281] Reducing Torch threads from 24 to 1 for serving. Set OMP_NUM_THREADS in the external environment to override.
(EngineCore pid=32543) INFO 11-14 04:07:01 [utils.py:381] Using LBNHC KV cache layout.
(EngineCore pid=32543) INFO 11-14 04:07:03 [monitor.py:40] [backbone] Starting torch.compile
(EngineCore pid=32543) INFO 11-14 04:07:25 [backends.py:1125] [backbone] Using cache directory: /root/.cache/vllm/torch_compile_cache/28c97495a4/rank_0_0/backbone for vLLM's torc

2026-11-14 04:07:50,259 - INFO - core.py:594 - flashinfer.jit: Building JIT module sampling; this can take several minutes on first use.
Capturing CUDA graphs (FULL): 100%|██████████| 2/2 [00:00<00:00,  7.30it/s]


(EngineCore pid=32543) INFO 11-14 04:09:07 [model_runner.py:1163] Graph capturing finished in 28 secs, took 0.71 GiB
(EngineCore pid=32543) INFO 11-14 04:09:08 [gpu_worker.py:772] Available KV cache memory: 12.73 GiB
(EngineCore pid=32543) INFO 11-14 04:09:08 [gpu_worker.py:787] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.9000 is equivalent to --gpu-memory-utilization=0.8909 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.9091. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=32543) INFO 11-14 04:09:08 [kv_cache_utils.py:1147] GPU KV cache size: 41,712 tokens, Maximum concurrency for 8,192 tokens per request: 5.09x
(EngineCore pid=32543) INFO 11-14 04:09:08 [kernel_warmup.py:179] JIT kernel warmup starting.
(EngineCore pid=32543) INFO 11-14 04:09:08 [kernel_warmup.py:192] JIT kernel warmup finished in 0.00s.
(EngineCor

2026-11-14 04:09:08,468 - INFO - autotuner.py:1048 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
2026-11-14 04:09:08,986 - INFO - autotune_cache.py:495 - flashinfer.jit: [Autotuner]: Managed autotune cache attached for this process: ManagedAutotuneCache(env_dir='/root/.cache/flashinfer/autotune/v2/c0d05643f93ca890', hits=0, misses=0)


(EngineCore pid=32543) INFO 11-14 04:09:08 [kernel_warmup.py:371] Running FlashInfer autotune with 16384 tokens and token buckets (1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 768, 1024, 1280, 1536, 1792, 2048, 2560, 3072, 3584, 4096, 8192, 16384).


2026-11-14 04:09:31,751 - INFO - autotuner.py:1071 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (FULL): 100%|██████████| 51/51 [00:43<00:00,  1.17it/s]


(EngineCore pid=32543) INFO 11-14 04:10:28 [model_runner.py:1163] Graph capturing finished in 52 secs, took 0.32 GiB
(EngineCore pid=32543) INFO 11-14 04:10:28 [gpu_worker.py:1003] CUDA graph pool memory: 0.32 GiB (actual), 0.87 GiB (estimated), difference: 0.55 GiB (170.7%).
(EngineCore pid=32543) INFO 11-14 04:10:28 [gpu_worker.py:1066] Free memory on device (94.3/94.97 GiB) on startup. Desired GPU memory utilization is (0.9, 85.47 GiB). Actual usage is 68.31 GiB for consumed memory (weights + non-torch), 4.43 GiB for peak activation, and 0.32 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=13168055399` (12.26 GiB) to fit into requested memory, or `--kv-cache-memory=22641749504` (21.09 GiB) to fully utilize gpu memory. Current kv cache memory in use is 12.73 GiB.
(EngineCore pid=32543) INFO 11-14 04:10:29 [jit_monitor.py:84] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=32543) INFO

In [17]:
structured_params = StructuredOutputsParams(json=DocumentTables.model_json_schema())
sampling_params = SamplingParams(
    max_tokens=4096,
    structured_outputs=structured_params,
)

In [13]:
tokenizer = llm.get_tokenizer()
MAX_INPUT_TOKENS = 4096

def truncate_context(text: str, max_tokens: int = MAX_INPUT_TOKENS) -> str:
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    if len(token_ids) <= max_tokens:
        return text
    return tokenizer.decode(token_ids[:max_tokens], skip_special_tokens=True)

In [31]:
SYSTEM_PROMPT = """Extract all structured data tables from the provided text into the requested JSON schema.

Rules:
1. Split separate tables: Whenever a new set of column headers or a new section table appears, create a separate object in "tables" with a concise, descriptive "table_name" (e.g., "round_4_leaderboard").
2. Strict column alignment: Every list inside "rows" MUST have the exact same number of elements as "headers". If a cell is blank or omitted in the source text (such as tied ranks or missing scores), preserve alignment by inserting an empty string "".
3. Verbatim cell values: Copy cell text, numbers, dates, and symbols (e.g., "N/A", "T8", "+2", "63-71=134") exactly as written without reformatting or calculating.
4. Filter noise: Ignore Wikipedia navigation footers/boxes (e.g., "V*T*E ... [hide]", "v t e", "Categories:", "References") and narrative prose paragraphs. Include Wikipedia infoboxes as 2-column tables if present.
5. Empty fallback: If the text contains no structured data tables, return {"tables": []}."""

def run_resumable_extraction(
    all_unique_contexts: list[str],
    checkpoint_path: str = "/content/drive/MyDrive/wikidt_context_cache.parquet",
    batch_size: int = 250,
) -> dict[str, str]:
    # Resume from existing Google Drive Parquet checkpoint if present
    if os.path.exists(checkpoint_path):
        cache_df = pd.read_parquet(checkpoint_path)
        context_to_json = dict(zip(cache_df["context"], cache_df["context_tables_json"]))
        print(f"Resumed {len(context_to_json)} completed contexts from {checkpoint_path}")
    else:
        context_to_json = {}

    remaining = [c for c in all_unique_contexts if c not in context_to_json]
    print(f"Remaining unique contexts to extract: {len(remaining)}")

    for start in tqdm(range(0, len(remaining), batch_size), desc="Extracting batches"):
        batch = remaining[start : start + batch_size]
        conversations = [
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": truncate_context(ctx)},
            ]
            for ctx in batch
        ]
        # Continuous batching handles variable prompt/output lengths concurrently
        outputs = llm.chat(conversations, sampling_params=sampling_params)

        for ctx, out in zip(batch, outputs):
            choice = out.outputs[0]
            if choice.finish_reason == "stop":
                parsed = DocumentTables.model_validate_json(choice.text)
                context_to_json[ctx] = parsed.model_dump_json()
            else:
                # Store empty schema sentinel for rare oversized edge cases so they don't loop
                context_to_json[ctx] = '{"tables":[]}'

        # Persist checkpoint to Drive after every batch
        pd.DataFrame(
            list(context_to_json.items()),
            columns=["context", "context_tables_json"],
        ).to_parquet(checkpoint_path, index=False)

    return context_to_json

In [32]:
def linearize_from_json(tables_json: str, include_table_names: bool = True) -> str:
    if not tables_json:
        return ""
    data = json.loads(tables_json)
    serialized_tables = []
    for t in data.get("tables", []):
        header_str = " | ".join(t["headers"]).lower()
        row_strs = [
            f"row {idx + 1} : " + " | ".join(row).lower()
            for idx, row in enumerate(t["rows"])
        ]
        body = f"col : {header_str} " + " ".join(row_strs)
        if include_table_names:
            serialized_tables.append(f"{t['table_name'].lower()} : {body}")
        else:
            serialized_tables.append(body)
    return " ".join(serialized_tables)

def attach_tabular_columns(df: pd.DataFrame, context_to_json: dict[str, str]) -> pd.DataFrame:
    out_df = df.copy()
    out_df["context_tables_json"] = out_df["context"].map(context_to_json).fillna('{"tables":[]}')
    out_df["num_tables"] = out_df["context_tables_json"].apply(
        lambda x: len(json.loads(x).get("tables", []))
    )

    out_df["context_multitabqa"] = out_df["context_tables_json"].apply(
        lambda x: linearize_from_json(x, include_table_names=True)
    )

    out_df["context_tapex"] = out_df["context_tables_json"].apply(
        lambda x: linearize_from_json(
            x, include_table_names=(len(json.loads(x).get("tables", [])) > 1)
        )
    )
    return out_df


all_unique = (
    pd.concat([train_df["context"], val_df["context"], test_df["context"]])
    .dropna()
    .unique()
    .tolist()
)
context_cache = run_resumable_extraction(all_unique)

train_processed = attach_tabular_columns(train_df, context_cache)
val_processed = attach_tabular_columns(val_df, context_cache)
test_processed = attach_tabular_columns(test_df, context_cache)

Remaining unique contexts to extract: 16638


Extracting batches:   0%|          | 0/67 [00:00<?, ?it/s]

Rendering conversations:   0%|          | 0/250 [00:00<?, ?it/s]


Processed prompts: 100%|██████████| 250/250 [16:05<00:00,  3.86s/it, est. speed input: 269.20 toks/s, output: 309.16 toks/s]


Rendering conversations:   0%|          | 0/250 [00:00<?, ?it/s]


Processed prompts: 100%|██████████| 250/250 [14:21<00:00,  3.45s/it, est. speed input: 289.96 toks/s, output: 323.22 toks/s]


... (126 outputs left out)

Rendering conversations:   0%|          | 0/250 [00:00<?, ?it/s]


Processed prompts: 100%|██████████| 250/250 [16:36<00:00,  3.99s/it, est. speed input: 272.00 toks/s, output: 287.58 toks/s]


Rendering conversations:   0%|          | 0/138 [00:00<?, ?it/s]


Processed prompts: 100%|██████████| 138/138 [10:54<00:00,  4.74s/it, est. speed input: 227.89 toks/s, output: 246.84 toks/s]


In [53]:
# Save final enriched splits to Drive
train_processed.to_parquet("/content/drive/MyDrive/266_Final/data/Train/wikidt_train_tabular.parquet", index=False)
val_processed.to_parquet("/content/drive/MyDrive/266_Final/data/Val/wikidt_val_tabular.parquet", index=False)
test_processed.to_parquet("/content/drive/MyDrive/266_Final/data/Test/wikidt_test_tabular.parquet", index=False)

In [54]:
updated_context = load_dataset(
    "parquet",
    data_files={
        "train": "/content/drive/MyDrive/266_Final/data/Train/wikidt_train_tabular.parquet",
        "validation": "/content/drive/MyDrive/266_Final/data/Val/wikidt_val_tabular.parquet",
        "test": "/content/drive/MyDrive/266_Final/data/Test/wikidt_test_tabular.parquet",
    },
)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [55]:
updated_context_train_df = updated_context["train"].to_pandas()

In [56]:
updated_context_train_df.head()

,question,answer,context,pairs,positive_sentences,image,context_tables_json,num_tables,context_multitabqa,context_tapex
0,geno hayes and nick roach both played which po...,no answer,Position Player $ Free agency tag Date signed ...,"[{'label': 0, 'query': 'geno hayes and nick ro...",0,204_783_page.png,"{""tables"":[{""table_name"":""free_agents"",""header...",3,free_agents : col : position | player | $ | fr...,free_agents : col : position | player | $ | fr...
1,What is the number of bronze for Canada and si...,no answer,"Pos Team Time Gold Switzerland (Reto Götschi, ...","[{'label': 0, 'query': 'What is the number of ...",0,11338408_page.png,"{""tables"":[{""table_name"":""two_man_bobsleigh"",""...",4,two_man_bobsleigh : col : pos | team | time | ...,two_man_bobsleigh : col : pos | team | time | ...
2,"What is Date, when Opponent is ""Philadelphia 7...",february 16,# Date H/A/N Opponent W/L Score Record 40 Janu...,"[{'label': 1, 'query': 'What is Date, when Opp...",1,16946097_page.png,"{""tables"":[{""table_name"":""game_results"",""heade...",1,game_results : col : date | h/a/n | opponent |...,col : date | h/a/n | opponent | w/l | score | ...
3,Which Class has a Weight of 203?,sr,Position Number Name Height Weight Class Homet...,"[{'label': 1, 'query': 'Which Class has a Weig...",29,14624447_page.png,"{""tables"":[{""table_name"":""defensive_players"",""...",6,defensive_players : col : position | number | ...,defensive_players : col : position | number | ...
4,Who had 0 total votes in the purple team?,eliminated week 12 - the walk,Members Teammate Couples Team Blue Red 1 Blue ...,"[{'label': 1, 'query': 'Who had 0 total votes ...",1,18426579_page.png,"{""tables"":[{""table_name"":""castaway_elimination...",1,castaway_eliminations : col : members | teamma...,col : members | teammate | couples | team | bl...
